# Grafo de colaboración de Samuel L. Jackson
**Análisis y Ciencia de Datos — UNIS · Parte individual · Rodolfo Flores**

Sea $B$ la matriz actores × películas ($B_{am}=1$ si el actor $a$ sale en la película $m$). Entonces $A=BB^\top$
cuenta cuántas películas comparten dos actores: un grafo de colaboración es un **kernel**, igual que el del bosque
de la parte grupal (dos filas se parecen si caen en las mismas hojas).

El notebook corre de principio a fin **sin API key** usando el cache de `cache/`. Para reconstruir el cache hace falta
definir `TMDB_API_KEY` en el entorno (nunca se guarda en el repositorio).

In [1]:
from collections import Counter, defaultdict, deque
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

from grafo import Graph
from tmdb_utils import TMDBAPIUtils

pd.set_option("display.width", 140)
np.set_printoptions(precision=4, suppress=True)

## I.1 Cuenta y API key
Cuenta creada en themoviedb.org con API key de *developer*. La key se lee de la variable de entorno `TMDB_API_KEY`;
`.env` está en `.gitignore`.

## I.2 Clase `Graph`
Implementada en `grafo.py`: nodos `(id, nombre)`, aristas no dirigidas guardadas con la clave `(min, max)` para que
$(a,b)$ y $(b,a)$ sean la misma, y el peso sube en 1 cada vez que se agrega una arista que ya existe.

In [2]:
g_prueba = Graph()
for nodo in [(1, "a"), (2, "b"), (3, "c"), (1, "repetido")]:
    g_prueba.add_node(nodo)
g_prueba.add_edge(1, 2); g_prueba.add_edge(2, 1); g_prueba.add_edge(2, 3)
A_prueba, ids_prueba = g_prueba.adjacency()
print("nodos:", g_prueba.total_nodes(), "| aristas:", g_prueba.total_edges(), "| grado máximo:", g_prueba.max_degree_nodes())
print("orden de filas:", ids_prueba); print(A_prueba)
assert g_prueba.nodes[1] == "a" and A_prueba[0, 1] == 2 and (A_prueba == A_prueba.T).all()

nodos: 3 | aristas: 2 | grado máximo: {2: 2}
orden de filas: [1, 2, 3]
[[0. 2. 0.]
 [2. 0. 1.]
 [0. 1. 0.]]


## I.3 Clase `TMDBAPIUtils`
En `tmdb_utils.py`. Cada respuesta del API se guarda como JSON en `cache/` y se lee de ahí si ya existe.
* `get_movie_cast(movie_id, limit, exclude_ids)`: miembros del elenco con `order` entre 0 y `limit − 1`, sin `exclude_ids`.
* `get_movie_credits_for_person(person_id, start_date, end_date)`: `id`, `title`, `release_date` y `order` de cada
  película; filtramos por fecha nosotros, quitamos documentales (`genre_ids` contiene 99) y papeles donde la persona
  se interpreta a sí misma (personaje "Himself", "Herself" o "Self").

In [3]:
api = TMDBAPIUtils(cache_dir="cache")
SEMILLA_NOMBRE = "Samuel L. Jackson"
candidatos = api.buscar_persona(SEMILLA_NOMBRE)
SEMILLA = candidatos[0]["id"]
print("semilla:", candidatos[0]["name"], "| id TMDB:", SEMILLA, "| conocido por:", candidatos[0].get("known_for_department"))

semilla: Samuel L. Jackson | id TMDB: 2231 | conocido por: Acting


## I.4 Construcción del grafo
`limit = 5` y ventana del 1 de enero de 1975 al 31 de diciembre de 2025. Una película **cuenta** para un actor si salió
dentro de la ventana y el actor aparece con `order < 5`.

In [4]:
LIMIT = 5
INICIO, FIN = "1975-01-01", "2025-12-31"


def peliculas_que_cuentan(persona):
    return {c["id"]: c for c in api.get_movie_credits_for_person(persona, INICIO, FIN)
            if c["order"] is not None and c["order"] < LIMIT}


g = Graph()
# (a) la semilla
g.add_node((SEMILLA, SEMILLA_NOMBRE))
# (b) elenco principal de cada película que cuenta para la semilla
pelis_semilla = peliculas_que_cuentan(SEMILLA)
for pid in pelis_semilla:
    for miembro in api.get_movie_cast(pid, LIMIT, exclude_ids=[SEMILLA]):
        g.add_node((miembro["id"], miembro["name"]))
print(f"películas que cuentan para la semilla: {len(pelis_semilla)} | nodos tras (b): {g.total_nodes()}")

# (c) una arista por película y por pareja, entre nodos que ya están en el grafo
cuenta_para = {n: peliculas_que_cuentan(n) for n in g.nodes}
titulo = {}
nodos_por_peli = defaultdict(set)
for n, pelis in cuenta_para.items():
    for pid, c in pelis.items():
        nodos_por_peli[pid].add(n)
        titulo[pid] = f"{c['title']} ({c['release_date'][:4]})"
for pid, nodos in nodos_por_peli.items():
    for a, b in combinations(sorted(nodos), 2):
        g.add_edge(a, b)

A, ids = g.adjacency()
nombre = [g.nodes[i] for i in ids]
grado = g.degree()
print("nodos:", g.total_nodes(), "| aristas:", g.total_edges(), "| peso total:", int(A.sum() / 2))
print("nodo(s) de mayor grado:", {g.nodes[n]: d for n, d in g.max_degree_nodes().items()})
print("llamadas nuevas al API en esta corrida:", api.llamadas_api)

películas que cuentan para la semilla: 108 | nodos tras (b): 377
nodos: 377 | aristas: 2666 | peso total: 2984
nodo(s) de mayor grado: {'Samuel L. Jackson': 371}
llamadas nuevas al API en esta corrida: 0


In [5]:
top_grado = sorted(grado.items(), key=lambda t: -t[1])[:10]
pd.DataFrame([(g.nodes[n], d, int(A[ids.index(n)].sum())) for n, d in top_grado],
             columns=["actor", "grado", "grado ponderado"])

,actor,grado,grado ponderado
0,Samuel L. Jackson,371,415
1,Bruce Willis,54,71
2,Robert De Niro,45,56
3,Nicolas Cage,44,59
4,Julianne Moore,43,47
5,Danny Glover,42,48
6,Morgan Freeman,41,50
7,Ed Harris,39,42
8,Gary Oldman,39,45
9,Kevin Spacey,36,38


In [6]:
aristas_pesadas = sorted(g.edges.items(), key=lambda t: -t[1])[:10]
pd.DataFrame([(g.nodes[a], g.nodes[b], w) for (a, b), w in aristas_pesadas],
             columns=["actor 1", "actor 2", "películas juntos"])

,actor 1,actor 2,películas juntos
0,Vin Diesel,Michelle Rodriguez,8
1,Eddie Murphy,Antonio Banderas,8
2,Scarlett Johansson,Chris Evans,7
3,Bruce Willis,Samuel L. Jackson,5
4,Antonio Banderas,Salma Hayek Pinault,5
5,Chris Rock,Jada Pinkett Smith,5
6,Rupert Everett,Colin Firth,5
7,Natalie Portman,Ewan McGregor,4
8,Samuel L. Jackson,Hayden Christensen,4
9,Mark Strong,Colin Firth,4


**Resultado.** 108 películas cuentan para Samuel L. Jackson; con sus elencos principales el grafo tiene **377 actores y
2 666 aristas**. El nodo de mayor grado es la propia semilla (371 vecinos), lejos del segundo, Bruce Willis (54).
Las aristas más pesadas no involucran a la semilla: el paso (c) suma **todas** las películas que comparten dos actores
del grafo en la ventana, así que aparecen parejas de franquicias (Vin Diesel–Michelle Rodriguez en *Fast & Furious*,
Eddie Murphy–Antonio Banderas en *Shrek*, Chris Evans–Scarlett Johansson en Marvel).